In [36]:
import numpy as np
import pandas as pd

In [37]:
df = pd.read_csv('gurgaon_properties_post_feature_selection.csv')

In [38]:
df.head()

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category,price
0,0.0,40.0,3,2,2.0,1.0,850.0,0,0,1,1.0,1.0,0.82
1,0.0,101.0,2,2,2.0,1.0,1226.0,1,0,1,1.0,2.0,0.95
2,0.0,111.0,2,2,1.0,1.0,1000.0,0,0,1,1.0,0.0,0.32
3,0.0,105.0,3,4,4.0,3.0,1615.0,1,0,2,0.0,2.0,1.60
4,0.0,4.0,2,2,1.0,3.0,582.0,0,1,1,0.0,2.0,0.48


In [39]:
# one hot encode -> sector, balcony, agePossession, furnishing type, luxury category, floor category

In [40]:
X = df.drop(columns=['price'])
y = df['price']

In [41]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

In [42]:
columns_to_encode = ['sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [43]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

In [44]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(),
         ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),

        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'),
         ['property_type'] + columns_to_encode)
    ],
    remainder='passthrough'
)

In [45]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', SVR(kernel='rbf'))
])

In [28]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

C:\Users\simra\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [29]:
scores.mean()

np.float64(0.8843777062004223)

In [30]:
scores.std()

np.float64(0.016357696271683344)

In [31]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [32]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [33]:
y_pred = pipeline.predict(X_test)

C:\Users\simra\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [34]:
y_pred = np.expm1(y_pred)

In [46]:
y_pred

array([ 1.6118409 ,  0.66988034,  1.08650825,  4.57068628,  2.48187205,
        4.2440186 ,  8.85493569,  1.61649773,  1.6505573 ,  1.62602647,
        3.09268922,  1.03506629,  2.83997287,  1.46037309, 11.58428766,
        0.73703878,  1.52610531,  0.25855095,  1.07803038,  2.75935962,
        1.42234723,  1.41930951,  0.69976261,  0.95826847,  1.74323861,
        3.22203339,  0.81973495,  0.88736466,  1.17879921, 11.76440436,
        0.58654455,  0.56001468,  4.66715872,  0.3803481 ,  1.14883064,
        4.7380612 ,  1.58426689,  0.6303807 ,  0.60634077,  0.52962286,
        0.74396856,  1.66904949,  0.6030045 ,  2.1970275 ,  1.33535303,
        0.50806657,  8.06855767,  1.95209745,  2.5275239 ,  1.59915532,
        1.62461108,  0.75521013,  5.9230844 ,  2.27407964,  1.04307725,
        0.76681934,  7.4892386 ,  4.39755761,  5.35996549,  1.59403028,
        1.85154272,  2.86735693,  0.63280581,  2.1752091 ,  0.29837202,
        1.6178337 ,  1.79007783,  2.45617326,  0.63213285,  1.15

In [35]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.expm1(y_test),y_pred)

0.5259504076563809